In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import numpy as np
import os
import zipfile # To handle zip files
import shutil # To remove directories

# --- 1. 🛠️ USER CONFIGURATION ---
IMG_WIDTH = 64      # Target width for resizing images (adjust as needed)
IMG_HEIGHT = 64     # Target height for resizing images (adjust as needed)
IMAGE_SIZE = (IMG_WIDTH, IMG_HEIGHT)
COLOR_MODE = 'grayscale' # 'grayscale' or 'rgb'. Grayscale is often fine for alphabet images.
CHANNELS = 1 if COLOR_MODE == 'grayscale' else 3

# General training parameters
BATCH_SIZE = 32
EPOCHS = 20         # Adjust (15-30 is a good start for ANNs on simple datasets)
LEARNING_RATE = 0.001

# Dataset parameters
DATASET_ZIP_NAME = 'alphabet_dataset.zip' # The name of the zip file you will upload
DATASET_EXTRACT_PATH = 'extracted_ann_dataset'

# --- 2. 📂 Dataset Upload and Preparation ---
print(f"🚀 Starting ANN Image Classification for Alphabets")
print(f"Expecting images of size: {IMAGE_SIZE}, Color: {COLOR_MODE}")

from google.colab import files
print(f"\nPlease upload your dataset ZIP file named '{DATASET_ZIP_NAME}'")
uploaded = files.upload()

if DATASET_ZIP_NAME in uploaded:
    print(f"\n✅ '{DATASET_ZIP_NAME}' uploaded successfully!")
    if os.path.exists(DATASET_EXTRACT_PATH):
        print(f"🧹 Cleaning up existing directory: {DATASET_EXTRACT_PATH}")
        shutil.rmtree(DATASET_EXTRACT_PATH)
    os.makedirs(DATASET_EXTRACT_PATH, exist_ok=True)

    with zipfile.ZipFile(DATASET_ZIP_NAME, 'r') as zip_ref:
        zip_ref.extractall(DATASET_EXTRACT_PATH)
    print(f"🗂️ Dataset extracted to '{DATASET_EXTRACT_PATH}'")

    extracted_items = os.listdir(DATASET_EXTRACT_PATH)
    if not extracted_items:
        print(f"❌ Error: The extracted directory '{DATASET_EXTRACT_PATH}' is empty.")
        exit()

    dataset_dir = DATASET_EXTRACT_PATH
    if len(extracted_items) == 1 and os.path.isdir(os.path.join(DATASET_EXTRACT_PATH, extracted_items[0])):
        dataset_dir = os.path.join(DATASET_EXTRACT_PATH, extracted_items[0])

    print(f"🔍 Using dataset directory: {dataset_dir}")

    class_folders = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
    if not class_folders or len(class_folders) < 2:
        print(f"❌ ERROR: Dataset directory '{dataset_dir}' must contain at least two subfolders (classes).")
        exit()
else:
    print(f"❌ ERROR: '{DATASET_ZIP_NAME}' not found. Please upload the correct file.")
    exit()

# --- 3. 🖼️ Load Data with Preprocessing ---
print("\n⏳ Loading and preprocessing data...")

try:
    # Load images, resize, and convert to specified color mode
    train_dataset = image_dataset_from_directory(
        dataset_dir,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMAGE_SIZE,
        color_mode=COLOR_MODE, # 'grayscale' or 'rgb'
        batch_size=BATCH_SIZE,
        label_mode='categorical' # For categorical_crossentropy
    )

    validation_dataset = image_dataset_from_directory(
        dataset_dir,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMAGE_SIZE,
        color_mode=COLOR_MODE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("Ensure your dataset_dir contains subdirectories for each class and images are valid.")
    exit()

class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Found classes: {class_names} (Number of classes: {num_classes})")

if num_classes < 2:
    print(f"❌ Error: Classification requires at least 2 classes. Found only {num_classes}.")
    exit()

# Preprocessing function: Normalize pixel values to [0, 1]
def preprocess_ann_images(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Normalize to [0,1]
    return image, label

print("Applying normalization...")
train_dataset = train_dataset.map(preprocess_ann_images, num_parallel_calls=tf.data.AUTOTUNE)
validation_dataset = validation_dataset.map(preprocess_ann_images, num_parallel_calls=tf.data.AUTOTUNE)

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# --- 4. 🧠 Build the ANN Model ---
# For an ANN (MLP), we need to flatten the image data.
print(f"\n⏳ Building ANN model...")
model = models.Sequential([
    layers.Flatten(input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)), # Flatten the images
    layers.Dense(128, activation='relu'),               # First dense hidden layer
    layers.Dropout(0.3),                                # Dropout for regularization
    layers.Dense(64, activation='relu'),                # Second dense hidden layer
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')     # Output layer with softmax
])

# --- 5. ⚙️ Compile the Model ---
print("\n⚙️ Compiling the model...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy', # Use 'categorical_crossentropy' for one-hot encoded labels
    metrics=['accuracy']
)
model.summary()

# --- 6. 🚀 Train the Model ---
print("\n🚀 Starting model training...")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    verbose=1
)

# --- 7. 📊 Evaluate and Plot ---
print("\n⚖️ Evaluating model and plotting history...")
val_loss, val_accuracy = model.evaluate(validation_dataset, verbose=0)
print(f"\n✅ Final Validation Loss: {val_loss:.4f}")
print(f"🎯 Final Validation Accuracy: {val_accuracy*100:.2f}%")

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('ANN - Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('ANN - Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.show()

# --- 8. 🔮 Predict on a New Image ---
def predict_single_new_image_ann(trained_model, class_names_list, img_size_tuple, color_mode_str):
    print("\n🖼️ Upload an image for prediction:")
    uploaded_img_dict = files.upload()

    if not uploaded_img_dict:
        print("No file uploaded.")
        return

    file_path = list(uploaded_img_dict.keys())[0]

    try:
        # Load and preprocess the image exactly as done for training
        img = tf.keras.preprocessing.image.load_img(
            file_path,
            target_size=img_size_tuple,
            color_mode=color_mode_str
        )
        img_for_display = tf.keras.preprocessing.image.img_to_array(img) # For display

        img_for_model_input = tf.keras.preprocessing.image.img_to_array(img)
        img_for_model_input = tf.cast(img_for_model_input, tf.float32) / 255.0 # Normalize
        img_batch = tf.expand_dims(img_for_model_input, 0) # Create a batch

        predictions_output = trained_model.predict(img_batch)
        predicted_index = np.argmax(predictions_output[0])
        confidence = np.max(predictions_output[0]) * 100
        predicted_class_name = class_names_list[predicted_index]

        # Displaying the image
        plt.figure()
        if color_mode_str == 'grayscale':
            plt.imshow(img_for_display.squeeze(), cmap='gray') # Squeeze for grayscale if it has an extra dim
        else:
            plt.imshow(img_for_display.astype(np.uint8))
        plt.title(f"Predicted: {predicted_class_name} ({confidence:.2f}%)")
        plt.axis("off")
        plt.show()
        print(f"The image is predicted as: '{predicted_class_name}' with {confidence:.2f}% confidence.")

    except Exception as e:
        print(f"❌ Error processing or predicting image: {e}")

predict_single_new_image_ann(model, class_names, IMAGE_SIZE, COLOR_MODE)

print("\n🎉 --- End of ANN Classification Script --- 🎉")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import numpy as np
import os
import zipfile # To handle zip files
import shutil # To remove directories

# --- 1. 🛠️ USER CONFIGURATION ---
IMG_WIDTH = 64      # Target width for resizing images (adjust as needed for your dataset)
IMG_HEIGHT = 64     # Target height for resizing images (adjust as needed)
IMAGE_SIZE = (IMG_WIDTH, IMG_HEIGHT)
COLOR_MODE = 'grayscale' # 'grayscale' or 'rgb'. Grayscale often better for simpler ANNs & less data.
CHANNELS = 1 if COLOR_MODE == 'grayscale' else 3

# General training parameters
BATCH_SIZE = 32
EPOCHS = 25         # Adjust (15-30 is a good start for ANNs on simple datasets)
LEARNING_RATE = 0.001

# Dataset parameters
# For a dog vs. cat example, you might name your zip file 'dog_vs_cat_dataset.zip'
DATASET_ZIP_NAME = 'binary_classification_dataset.zip' # The name of the zip file you will upload
DATASET_EXTRACT_PATH = 'extracted_ann_binary_dataset'

# --- 2. 📂 Dataset Upload and Preparation ---
print(f"🚀 Starting ANN Image Classification (Suitable for Binary or Simple Multi-Class)")
print(f"Expecting images of size: {IMAGE_SIZE}, Color: {COLOR_MODE}")

from google.colab import files
print(f"\nPlease upload your dataset ZIP file named '{DATASET_ZIP_NAME}'")
print("This ZIP file should contain folders for each class (e.g., a 'dogs' folder and a 'cats' folder).")
uploaded = files.upload()

if DATASET_ZIP_NAME in uploaded:
    print(f"\n✅ '{DATASET_ZIP_NAME}' uploaded successfully!")
    if os.path.exists(DATASET_EXTRACT_PATH):
        print(f"🧹 Cleaning up existing directory: {DATASET_EXTRACT_PATH}")
        shutil.rmtree(DATASET_EXTRACT_PATH)
    os.makedirs(DATASET_EXTRACT_PATH, exist_ok=True)

    with zipfile.ZipFile(DATASET_ZIP_NAME, 'r') as zip_ref:
        zip_ref.extractall(DATASET_EXTRACT_PATH)
    print(f"🗂️ Dataset extracted to '{DATASET_EXTRACT_PATH}'")

    extracted_items = os.listdir(DATASET_EXTRACT_PATH)
    if not extracted_items:
        print(f"❌ Error: The extracted directory '{DATASET_EXTRACT_PATH}' is empty.")
        exit()

    dataset_dir = DATASET_EXTRACT_PATH
    # If the zip file contains a single root folder, navigate into it
    if len(extracted_items) == 1 and os.path.isdir(os.path.join(DATASET_EXTRACT_PATH, extracted_items[0])):
        dataset_dir = os.path.join(DATASET_EXTRACT_PATH, extracted_items[0])

    print(f"🔍 Using dataset directory: {dataset_dir}")

    class_folders = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
    if not class_folders or len(class_folders) < 2: # Need at least 2 classes for classification
        print(f"❌ ERROR: Dataset directory '{dataset_dir}' must contain at least two subfolders (classes). "
              "For binary classification (e.g., dogs vs cats), create a 'dogs' folder and a 'cats' folder.")
        exit()
else:
    print(f"❌ ERROR: '{DATASET_ZIP_NAME}' not found. Please upload the correct file.")
    exit()

# --- 3. 🖼️ Load Data with Preprocessing ---
print("\n⏳ Loading and preprocessing data...")

try:
    # Load images, resize, and convert to specified color mode
    train_dataset = image_dataset_from_directory(
        dataset_dir,
        validation_split=0.2,
        subset="training",
        seed=123, # Seed for shuffling and splitting consistency
        image_size=IMAGE_SIZE,
        color_mode=COLOR_MODE,
        batch_size=BATCH_SIZE,
        label_mode='categorical' # Generates one-hot encoded labels (e.g., [1,0], [0,1] for 2 classes)
    )

    validation_dataset = image_dataset_from_directory(
        dataset_dir,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMAGE_SIZE,
        color_mode=COLOR_MODE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("Ensure your dataset_dir contains subdirectories for each class and images are valid.")
    exit()

class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Found classes: {class_names} (Number of classes: {num_classes})")

if num_classes < 2: # This check is important
    print(f"❌ Error: Classification requires at least 2 classes. Found only {num_classes}.")
    exit()

# Preprocessing function: Normalize pixel values to [0, 1]
def preprocess_ann_images(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Normalize to [0,1]
    return image, label

print("Applying normalization...")
train_dataset = train_dataset.map(preprocess_ann_images, num_parallel_calls=tf.data.AUTOTUNE)
validation_dataset = validation_dataset.map(preprocess_ann_images, num_parallel_calls=tf.data.AUTOTUNE)

# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# --- 4. 🧠 Build the ANN Model ---
# For an ANN (MLP), we need to flatten the image data.
print(f"\n⏳ Building ANN model...")
model = models.Sequential([
    layers.Flatten(input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)), # Flatten the images
    layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)), # Added L2 regularization
    layers.Dropout(0.4),                                # Increased Dropout
    layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),                 # Second dense hidden layer
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')     # Output layer with softmax
                                                        # For 2 classes, this outputs two probabilities
])

# --- 5. ⚙️ Compile the Model ---
print("\n⚙️ Compiling the model...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy', # Works for 2+ classes with one-hot encoded labels
    metrics=['accuracy']
)
model.summary() # Print model structure

# --- 6. 🚀 Train the Model ---
print("\n🚀 Starting model training...")
# Optional: Add early stopping
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    verbose=1
    # callbacks=[early_stopping] # Uncomment to use early stopping
)

# --- 7. 📊 Evaluate and Plot ---
print("\n⚖️ Evaluating model and plotting history...")
# If early stopping restored best weights, evaluation is on the best model.
# Otherwise, it's on the last epoch's model.
val_loss, val_accuracy = model.evaluate(validation_dataset, verbose=0)
print(f"\n✅ Final Validation Loss: {val_loss:.4f}")
print(f"🎯 Final Validation Accuracy: {val_accuracy*100:.2f}%")

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc)) # Use actual number of epochs run

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('ANN - Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('ANN - Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.show()

# --- 8. 🔮 Predict on a New Image ---
def predict_single_new_image_ann(trained_model, class_names_list, img_size_tuple, color_mode_str):
    print("\n🖼️ Upload an image for prediction:")
    uploaded_img_dict = files.upload()

    if not uploaded_img_dict:
        print("No file uploaded.")
        return

    file_path = list(uploaded_img_dict.keys())[0]

    try:
        # Load and preprocess the image exactly as done for training
        img = tf.keras.preprocessing.image.load_img(
            file_path,
            target_size=img_size_tuple,
            color_mode=color_mode_str
        )
        img_for_display = tf.keras.preprocessing.image.img_to_array(img) # For display

        # Preprocess for model input
        img_for_model_input = tf.keras.preprocessing.image.img_to_array(img)
        img_for_model_input = tf.cast(img_for_model_input, tf.float32) / 255.0 # Normalize
        img_batch = tf.expand_dims(img_for_model_input, 0) # Create a batch (shape: [1, height, width, channels])

        predictions_output = trained_model.predict(img_batch) # Output shape: (1, num_classes)

        # For softmax output with num_classes (even if num_classes is 2)
        predicted_index = np.argmax(predictions_output[0]) # Index of the highest probability
        confidence = np.max(predictions_output[0]) * 100    # Highest probability
        predicted_class_name = class_names_list[predicted_index]

        # Displaying the image
        plt.figure()
        if color_mode_str == 'grayscale':
            # Squeeze to remove channel dim for grayscale if it's (H, W, 1)
            plt.imshow(img_for_display.squeeze(), cmap='gray')
        else:
            plt.imshow(img_for_display.astype(np.uint8)) # Convert to uint8 for RGB display
        plt.title(f"Predicted: {predicted_class_name} ({confidence:.2f}%)")
        plt.axis("off")
        plt.show()
        print(f"The image is predicted as: '{predicted_class_name}' with {confidence:.2f}% confidence.")
        print(f"Raw prediction scores (probabilities for each class {class_names}): {predictions_output[0]}")


    except Exception as e:
        print(f"❌ Error processing or predicting image: {e}")

# Make a prediction on a new image using the trained ANN model
predict_single_new_image_ann(model, class_names, IMAGE_SIZE, COLOR_MODE)

print("\n🎉 --- End of ANN Classification Script --- 🎉")